# Phân tích Sentiment IMDB với Embedding & TensorBoard Projector

Notebook này hướng dẫn tải dữ liệu IMDB bằng Pandas, chuyển đổi thành `tf.data.Dataset`, xây dựng từ vựng bằng `TextVectorization`, huấn luyện mô hình phân loại nhị phân và trực quan hóa các vector Embedding bằng TensorBoard Projector.

In [ ]:
import tensorflow as tf
import pandas as pd
import os
from tensorflow.keras.layers import TextVectorization
from tensorboard.plugins import projector

# 1. Tải và đọc dữ liệu IMDB bằng Pandas (tránh lỗi của thư viện tensorflow_datasets)
url = "https://storage.googleapis.com/protonx-cloud-storage/datasets/IMDB%20Dataset.csv"
csv_path = "IMDB Dataset.csv"

if not os.path.exists(csv_path):
    print("Đang tải dữ liệu IMDB...")
    # Tải file dùng Keras utility lưu vào thư mục hiện tại
    tf.keras.utils.get_file(csv_path, origin=url, cache_dir=".", cache_subdir="")

df = pd.read_csv(csv_path)

# Chuyển nhãn text ('positive'/'negative') thành số (1/0)
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

# Tạo tf.data.Dataset từ Pandas DataFrame
dataset = tf.data.Dataset.from_tensor_slices((
    df['review'].values,
    df['sentiment'].values
))

# Phân chia tập dữ liệu (80% train, 20% test)
DATASET_SIZE = len(df)
train_size = int(0.8 * DATASET_SIZE)

train_data = dataset.take(train_size)
test_data = dataset.skip(train_size)

In [ ]:
# 2. Xem thử một mẫu dữ liệu thô
for text_batch, label_batch in train_data.take(1):
    print("Review thô:", text_batch.numpy().decode('utf-8'))
    print("Label (0: Negative, 1: Positive):", label_batch.numpy())

In [ ]:
# 3. Cấu hình TextVectorization chuyển chữ thành số
vecto_data = TextVectorization(
    max_tokens=5000,
    output_sequence_length=200
)

# Chỉ lấy văn bản (bỏ qua nhãn) để adapt xây dựng từ điển (vocabulary)
train_text = train_data.map(lambda text, label: text)
vecto_data.adapt(train_text)

In [ ]:
# 4. Áp dụng TextVectorization lên toàn bộ dữ liệu và tạo các batch học
train_data = train_data.map(lambda text, label: (vecto_data(text), label))
test_data = test_data.map(lambda text, label: (vecto_data(text), label))

# Shuffle và Batching dữ liệu
train_data = train_data.shuffle(10000).batch(64).prefetch(tf.data.AUTOTUNE)
test_data = test_data.batch(64).prefetch(tf.data.AUTOTUNE)

# Kiểm tra kích thước của 1 batch
x, y = next(iter(train_data))
print("Kích thước Tensor đầu vào (X):", x.shape)  # (batch_size, sequence_length)
print("Kích thước Tensor nhãn (y):", y.shape)       # (batch_size,)

In [ ]:
# 5. Xây dựng mô hình phân loại nhị phân
model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=5000,   # Bằng max_tokens của TextVectorization
        output_dim=16     # Chiều vector biểu diễn của mỗi từ
    ),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

In [ ]:
# 6. Biên dịch mô hình
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
# 7. Huấn luyện mô hình
history = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data
)

In [ ]:
# 8. Dự đoán thử một câu tùy ý
sample = ['This movie was absolutely amazing and wonderful!']
encode = vecto_data(sample)
prediction = model.predict(encode)[0][0]

print(f"Xác suất Positive: {prediction:.4f}")
if prediction > 0.5:
    print("Dự đoán: Positive (Tích cực)")
else:
    print("Dự đoán: Negative (Tiêu cực)")

## Cấu hình lưu trữ dữ liệu Embedding để trực quan hóa

Ở đây chúng ta sẽ tạo các file metadata chứa danh sách từ vựng, lưu checkpoint các trọng số của lớp Embedding để tải lên công cụ trực quan hóa (projector).

In [ ]:
# 9. Tạo thư mục lưu log và ghi nhãn metadata cho từ vựng
log_dir = "logs"
os.makedirs(log_dir, exist_ok=True)

vocab = vecto_data.get_vocabulary()

# Lưu ý: Chúng ta ghi từ index 1 trở đi (vocab[1:]) để khớp với việc loại bỏ
# trọng số ở hàng đầu tiên (padding/UNK) ở bước lưu checkpoint tiếp theo.
# Điều này giúp tránh lỗi Dimension Mismatch (lệch số lượng nhãn so với số lượng vector) trên TensorBoard.
with open(os.path.join(log_dir, "metadata.tsv"), "w", encoding="utf-8") as f:
    for token in vocab[1:]:
        f.write(token + "\n")

In [ ]:
# 10. Lấy trọng số của lớp Embedding (bỏ index 0 chứa padding) và lưu checkpoint
weights = tf.Variable(model.layers[0].get_weights()[0][1:])
checkpoint = tf.train.Checkpoint(embedding=weights)
checkpoint.save(os.path.join(log_dir, "embedding.ckpt"))

In [ ]:
# 11. Cấu hình ProjectorConfig để chỉ đường dẫn metadata cho TensorBoard
config = projector.ProjectorConfig()
embedding = config.embeddings.add()
embedding.tensor_name = "embedding/.ATTRIBUTES/VARIABLE_VALUE"
embedding.metadata_path = 'metadata.tsv'
projector.visualize_embeddings(log_dir, config)

### Khởi chạy TensorBoard Projector trên Google Colab

Bạn có thể khởi chạy trực tiếp TensorBoard ngay trong notebook bằng các dòng lệnh dưới đây.

**Lưu ý:** Nếu bạn gặp lỗi màn hình đen hoặc không mở được cổng kết nối trên Colab do vấn đề bảo mật trình duyệt, bạn có thể tải 3 file kết quả về máy rồi upload trực tiếp lên trang web online chính thức của Tensorflow: [http://projector.tensorflow.org/](http://projector.tensorflow.org/) để xem trực quan hóa 3D rất mượt mà.

In [ ]:
# 12. Khởi chạy extension TensorBoard trong Colab
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
!zip -r logs.zip logs

  adding: logs/ (stored 0%)
  adding: logs/embedding.ckpt-1.index (deflated 28%)
  adding: logs/metadata.tsv (deflated 52%)
  adding: logs/checkpoint (deflated 40%)
  adding: logs/projector_config.pbtxt (deflated 13%)
  adding: logs/embedding.ckpt-1.data-00000-of-00001 (deflated 7%)


In [ ]:
ls

'IMDB Dataset.csv'   logs/   logs.zip   sample_data/


In [ ]:
# Tải file logs.zip lên dịch vụ file.io miễn phí
!curl -F "file=@logs.zip" https://file.io

<html>
<head><title>301 Moved Permanently</title></head>
<body>
<center><h1>301 Moved Permanently</h1></center>
<hr><center>cloudflare</center>
</body>
</html>
